# 02A · Campus Expense Agent
**Session 2 practice · 35–45 minutes · synthetic data only**

Build an assistant that can record a campus expense and explain the remaining
budget using Python tools. A model may propose a tool call; Python decides whether
the arguments are valid and performs the arithmetic.

**Problem:** starting with ₹120 food and ₹40 transport against a ₹500 budget,
add ₹80 food and prove the ledger contains three expenses totalling ₹240.
Unknown categories, invalid money and invented tools must leave state unchanged.

**Run all first.** The default `TEST_FIXTURE` mode uses visible scripted tool
decisions, not AI. It needs no account, key, installation or network. Then complete
the three student changes near the end. Optional live Gemini sends only the
synthetic prompt and tool observations to your own provider account.


## 1. Configure the offline baseline
Keep live execution off for the first run. Amounts are in INR; stored money uses integer paise.


In [ ]:
# The default path uses explicit scripted decisions and makes no network requests.
USE_LIVE_MODEL = False
MODEL = "gemini-3.5-flash-lite"
# Verify this model and account access with the instructor before the live exercise.
REQUEST_TIMEOUT_MS = 60_000
MAX_REQUESTS, MAX_STEPS, MAX_TOOLS_PER_TURN = 24, 8, 12
# Standard-library imports are sufficient for every offline baseline cell.
import copy
import json
import math
import time
from decimal import Decimal
print("Mode:", "LIVE_MODEL (optional)" if USE_LIVE_MODEL else "TEST_FIXTURE (scripted, not AI)")


## 2. Define authoritative state and tools
Read the validation order before running. `add_expense(category, amount_inr)` can add one validated synthetic record. The summary always comes from stored state.


In [ ]:
CATEGORIES = ("food", "transport", "learning", "other")

class ExpenseState:
    # Expense additions are local simulations; event-style approval is not required here.
    write_tools = set()
    def __init__(self, budget_inr=500):
        self.budget_paise = int(Decimal(str(budget_inr)) * 100)
        self.reset()

    def reset(self):
        # Every record is synthetic; no bank, UPI or personal account is connected.
        self.expenses = [{"id": "E001", "category": "food", "amount_paise": 12_000},
                         {"id": "E002", "category": "transport", "amount_paise": 4_000}]

    def summary(self):
        totals = {category: 0 for category in CATEGORIES}
        for row in self.expenses:
            totals[row["category"]] += row["amount_paise"]
        total = sum(totals.values())
        return {"simulated": True, "count": len(self.expenses),
            "by_category_inr": {k: v / 100 for k, v in totals.items()},
            "total_inr": total / 100, "budget_inr": self.budget_paise / 100,
            "remaining_inr": (self.budget_paise - total) / 100,
            "over_budget": total > self.budget_paise}

    def execute(self, name, arguments):
        # Reject unknown tools and incorrect argument keys before reading any values.
        contracts = {"list_expenses": set(), "get_budget_summary": set(),
                     "add_expense": {"category", "amount_inr"}}
        if name not in contracts:
            return {"ok": False, "error": "UNKNOWN_TOOL"}
        if not isinstance(arguments, dict) or set(arguments) != contracts[name]:
            return {"ok": False, "error": "ARGUMENT_CONTRACT"}
        if name == "list_expenses":
            rows = [{"id": r["id"], "category": r["category"],
                     "amount_inr": r["amount_paise"] / 100} for r in self.expenses]
            return {"ok": True, "simulated": True, "expenses": rows}
        if name == "get_budget_summary":
            return {"ok": True, **self.summary()}
        category, amount = arguments["category"], arguments["amount_inr"]
        if not isinstance(category, str) or category not in CATEGORIES:
            return {"ok": False, "error": "UNKNOWN_CATEGORY"}
        # Booleans, strings, NaN and infinity are not valid monetary amounts.
        if type(amount) not in (int, float) or (type(amount) is float and not math.isfinite(amount)):
            return {"ok": False, "error": "FINITE_NUMBER_REQUIRED"}
        value = Decimal(str(amount))
        if not Decimal("0") < value <= Decimal("5000"):
            return {"ok": False, "error": "AMOUNT_RANGE_0_TO_5000"}
        paise = value * 100
        if paise != paise.to_integral_value():
            return {"ok": False, "error": "AT_MOST_TWO_DECIMAL_PLACES"}
        # Integer paise prevent cumulative floating-point errors in the ledger.
        row = {"id": f"E{len(self.expenses) + 1:03d}", "category": category,
               "amount_paise": int(paise)}
        self.expenses.append(row)
        return {"ok": True, "simulated": True, "added_id": row["id"], **self.summary()}

EXPENSE_SCHEMAS = [
    {"name": "list_expenses", "description": "Read the synthetic expense ledger.",
     "parameters": {"type": "OBJECT", "properties": {}}},
    {"name": "get_budget_summary", "description": "Read accurate category totals and remaining budget in INR.",
     "parameters": {"type": "OBJECT", "properties": {}}},
    {"name": "add_expense", "description": "Add a simulated expense in a known category, 0 < INR <= 5000, at most 2 decimals.",
     "parameters": {"type": "OBJECT", "properties": {
         "category": {"type": "STRING", "enum": list(CATEGORIES)},
         "amount_inr": {"type": "NUMBER"}}, "required": ["category", "amount_inr"]}},
]
EXPENSE_SYSTEM = """Help with a synthetic campus expense ledger. Use tools for every
expense addition, category fact and total. Never say a change succeeded without an ok
tool result. Ask if category or amount is ambiguous. Never invent a transaction.
Unknown category, invalid amount or rejected tool means no change occurred.
All money and records are simulated; there is no financial account connection."""


## 3. Separate decisions from execution
`FixtureSource` supplies explicit test decisions. `GeminiSource` lazily loads the pinned SDK and your secret only for an opted-in live request. No live failure is replaced by fixture text.


In [ ]:
class PendingResponse(RuntimeError):
    """A safe, readable stop that never invents a completed model response."""

def tool_decision(*calls):
    # A fixture explicitly supplies a decision; it does not interpret natural language.
    parts = [{"function_call": {"name": name, "args": args, "id": call_id}}
             for name, args, call_id in calls]
    return {"complete": True, "content": {"role": "model", "parts": parts}}

def fixture_final(text):
    # This labelled text is test data, never a substitute for a failed live answer.
    return {"complete": True, "content": {"role": "model", "parts": [{"text": text}]}}

class FixtureSource:
    mode = "TEST_FIXTURE"
    def __init__(self, decisions):
        self.decisions = iter(decisions)

    def generate(self, history, schemas, system):
        # Consume the next named fixture decision regardless of the prompt wording.
        try:
            return copy.deepcopy(next(self.decisions))
        except StopIteration:
            raise PendingResponse("Fixture decisions ended; inspect the test sequence.") from None

class GeminiSource:
    mode = "LIVE_MODEL"
    def __init__(self, model=MODEL):
        self.model, self.client, self.types = model, None, None

    def _connect(self):
        # Import/install and secret access happen only after an explicit live request.
        import importlib.metadata as metadata
        import subprocess
        import sys
        try:
            installed = metadata.version("google-genai")
        except metadata.PackageNotFoundError:
            installed = None
        if installed != "2.12.0":
            # Preserve active google-auth constraints required by this Colab runtime.
            from packaging.requirements import Requirement
            try:
                requirements = map(Requirement, metadata.requires("google-colab") or [])
                constraints = [str(r) for r in requirements
                    if r.name.lower().replace("_", "-") == "google-auth"
                    and (r.marker is None or r.marker.evaluate())]
            except metadata.PackageNotFoundError:
                constraints = []
            subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "google-genai==2.12.0", *constraints], check=True,
                capture_output=True, text=True)
        from google import genai
        from google.genai import types
        import getpass
        api_key = None
        try:
            from google.colab import userdata
            try:
                api_key = userdata.get("GEMINI_API_KEY")
            except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
                pass
        except ImportError:
            pass
        if not api_key:
            api_key = getpass.getpass("Your Gemini API key (hidden): ").strip()
        if not api_key:
            raise PendingResponse("Add your own GEMINI_API_KEY in Colab Secrets, then resume.")
        # Disable SDK retries; ToolAgent owns the single bounded transient retry.
        self.client = genai.Client(api_key=api_key, http_options=types.HttpOptions(
            timeout=REQUEST_TIMEOUT_MS, retry_options=types.HttpRetryOptions(attempts=1)))
        self.types = types
        del api_key

    def generate(self, history, schemas, system):
        if self.client is None:
            self._connect()
        types = self.types
        # Declarations describe tools; automatic Python execution stays disabled.
        config = types.GenerateContentConfig(
            system_instruction=system,
            tools=[types.Tool(function_declarations=[types.FunctionDeclaration(**s) for s in schemas])],
            automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
            http_options=types.HttpOptions(timeout=REQUEST_TIMEOUT_MS,
                retry_options=types.HttpRetryOptions(attempts=1)),
            max_output_tokens=1024)
        reply = self.client.models.generate_content(model=self.model,
            contents=copy.deepcopy(history), config=config)
        candidate = reply.candidates[0] if reply.candidates else None
        reason = getattr(candidate, "finish_reason", None)
        reason = getattr(reason, "value", reason)
        content = getattr(candidate, "content", None)
        # Retain all SDK content fields, including tool IDs and thought signatures.
        return {"complete": reason == "STOP", "content":
            content.model_dump(exclude_none=True) if content is not None else None}


## 4. Execute a bounded tool loop
The dispatcher validates tools in Python and returns observations to the decision source. A failed generation keeps the exact pending turn. `resume()` continues after recorded observations, so an accepted expense is not added again.


In [ ]:
class ToolAgent:
    def __init__(self, state, source, schemas, system, approve_write=None):
        self.state, self.source = state, source
        self.schemas, self.system = schemas, system
        self.approve_write = approve_write
        self.messages, self.trace = [], []
        self.requests_used, self.pending = 0, None
        self.last_result, self.last_issue = None, None
        self._last_start = None
        self._sleep, self._clock = time.sleep, time.monotonic

    @property
    def can_resume(self):
        return bool(self.pending and not self.pending["reset_only"]
            and self.pending["step"] < MAX_STEPS and self.requests_used < MAX_REQUESTS)

    def _request(self):
        # Retry only generation from this frozen history, never a completed tool action.
        payload = copy.deepcopy(self.messages)
        for attempt in (1, 2):
            if self.requests_used >= MAX_REQUESTS:
                raise PendingResponse("Request budget exhausted; state and trace are retained.")
            if self.source.mode == "LIVE_MODEL" and self._last_start is not None:
                self._sleep(max(0, self._last_start + 5 - self._clock()))
            self._last_start = self._clock()
            self.requests_used += 1
            try:
                decision = self.source.generate(payload, self.schemas, self.system)
            except PendingResponse:
                raise
            except Exception as error:
                code = getattr(error, "code", None)
                code = code if isinstance(code, int) else None
                # Authentication and rate/quota errors never trigger automatic retries.
                transient = code in {408, 500, 502, 503, 504}
                if code not in {400, 401, 403, 404, 429}:
                    transient |= isinstance(error, (TimeoutError, ConnectionError))
                    transient |= type(error).__name__ in {"ReadTimeout", "ConnectTimeout", "ConnectError"}
                self.last_issue = {"stage": "generation", "code": code,
                                   "type": type(error).__name__, "attempt": attempt}
                if transient and attempt == 1 and self.requests_used < MAX_REQUESTS:
                    print("Transient generation issue: retrying the same response once.")
                    self._sleep(2 if self.source.mode == "LIVE_MODEL" else 0)
                    continue
                if code == 429:
                    message = "Quota/rate limit: stop and review your account with the instructor."
                elif code in {401, 403}:
                    message = "Check your private key and account permissions with the instructor."
                elif code == 404:
                    message = "Verify the configured model and account access before continuing."
                else:
                    message = "Generation did not finish; inspect setup or service availability."
                raise PendingResponse(message + " State and trace are kept; no final answer is claimed.") from None
            self.last_issue = None
            return decision

    def run(self, question):
        if self.pending is not None:
            raise PendingResponse("An earlier turn is pending; resume it before a new request.")
        if not isinstance(question, str) or not question.strip() or len(question) > 1200:
            raise PendingResponse("Use one nonempty question up to 1200 characters.")
        self.last_result = None
        self.messages.append({"role": "user", "parts": [{"text": question}]})
        self.pending = {"question": question, "step": 0, "tools": 0, "reset_only": False}
        return self._continue()

    def resume(self):
        # Reuse existing messages, tool observations and counters after a failed response.
        if not self.can_resume:
            raise PendingResponse("No resumable turn; inspect limits and partial state before reset.")
        return self._continue()

    def _continue(self):
        pending = self.pending
        while pending["step"] < MAX_STEPS:
            decision = self._request()
            pending["step"] += 1
            content = decision.get("content") if isinstance(decision, dict) else None
            if not isinstance(decision, dict) or not decision.get("complete") or not isinstance(content, dict):
                raise PendingResponse("Incomplete decision: no tools from this response ran; turn is pending.")
            parts = content.get("parts") or []
            if content.get("role") != "model" or not isinstance(parts, list):
                raise PendingResponse("Unexpected decision shape; turn is pending.")
            calls = [p["function_call"] for p in parts if p.get("function_call")]
            if not calls:
                answer = "".join(p.get("text", "") for p in parts if not p.get("thought"))
                if not answer.strip():
                    raise PendingResponse("No final answer was returned; turn remains pending.")
                self.messages.append(copy.deepcopy(content))
                self.last_result, self.pending = answer, None
                return answer
            if pending["tools"] + len(calls) > MAX_TOOLS_PER_TURN:
                pending["reset_only"] = True
                raise PendingResponse("Tool limit reached before this batch; inspect the trace.")
            self.messages.append(copy.deepcopy(content))
            observations = []
            try:
                for call in calls:
                    pending["tools"] += 1
                    name, arguments = call.get("name"), call.get("args", {})
                    approval = None
                    if name in self.state.write_tools:
                        # The client controls consent; model text cannot approve a write.
                        approval = bool(self.approve_write and
                            self.approve_write(name, copy.deepcopy(arguments)) is True)
                        result = self.state.execute(name, arguments, approved=approval)
                    else:
                        result = self.state.execute(name, arguments)
                    record = {"step": pending["step"], "tool": name,
                        "arguments": copy.deepcopy(arguments), "result": copy.deepcopy(result)}
                    if approval is not None:
                        record["client_approved"] = approval
                    self.trace.append(record)
                    response = {"name": name, "response": result}
                    if call.get("id"):
                        response["id"] = call["id"]
                    observations.append({"function_response": response})
            except Exception:
                # A partial batch must never be replayed after an unexpected tool failure.
                pending["reset_only"] = True
                raise PendingResponse("Tool batch interrupted. Inspect retained partial state; this batch cannot resume.") from None
            self.messages.append({"role": "user", "parts": observations})
        pending["reset_only"] = True
        raise PendingResponse("Step limit reached; the task is incomplete. Inspect the trace.")

    def show_trace(self):
        print(json.dumps(self.trace, indent=2))

    def reset(self):
        # Explicit reset clears the simulation, while preserving the consumed request budget.
        self.state.reset()
        self.messages.clear()
        self.trace.clear()
        self.pending, self.last_result, self.last_issue = None, None, None

def show_attempt(label, operation):
    # Expected classroom stops are readable output instead of red notebook tracebacks.
    try:
        answer = operation()
    except PendingResponse as error:
        print(label + ": PENDING — " + str(error))
        return None
    print(label + ": " + answer)
    return answer


## 5. Run and inspect the provided baseline
The fixture chooses summary → add expense → summary. Expected stored state: food ₹200, transport ₹40, total ₹240, remaining ₹260. Read the tool trace before the labelled fixture sentence.


In [ ]:
# Three scripted tool choices exercise the same dispatch boundary as live choices.
expense_fixture = FixtureSource([
    tool_decision(("get_budget_summary", {}, "expense-read-1")),
    tool_decision(("add_expense", {"category": "food", "amount_inr": 80}, "expense-add-1")),
    tool_decision(("get_budget_summary", {}, "expense-read-2")),
    fixture_final("TEST_FIXTURE final: the tool reports 3 expenses, total INR 240, remaining INR 260."),
])
expense_state = ExpenseState()
expense_agent = ToolAgent(expense_state, expense_fixture, EXPENSE_SCHEMAS, EXPENSE_SYSTEM)
baseline_reply = show_attempt("Expense baseline", lambda: expense_agent.run(
    "Read my budget, add INR 80 for food, then show the updated summary."))
expense_agent.show_trace()
# Check stored Python state rather than trusting the supplied final fixture sentence.
# New allowed categories start at zero; fixture amounts remain independent expected data.
expected_category_totals = {category: 0.0 for category in CATEGORIES}
for category, amount_inr in [("food", 120.0), ("transport", 40.0), ("food", 80.0)]:
    expected_category_totals[category] += amount_inr
expected_summary = {"simulated": True, "count": 3,
    "by_category_inr": expected_category_totals,
    "total_inr": 240.0, "budget_inr": 500.0, "remaining_inr": 260.0, "over_budget": False}
assert expense_state.summary() == expected_summary
assert baseline_reply is not None and expense_agent.pending is None
assert [r["tool"] for r in expense_agent.trace] == ["get_budget_summary", "add_expense", "get_budget_summary"]
print("PASS: stored totals and the 3-tool trace match the baseline.")


## 6. Prove rejected requests cannot edit the ledger
These are direct Python contract checks, separate from model decisions. Every expected rejection prints a readable result and checks the before/after ledger.


In [ ]:
# Use a separate ledger so rejection checks cannot change the completed baseline.
validation_state = ExpenseState()
rejection_cases = [
    ("add_expense", {"category": "shopping", "amount_inr": 10}, "UNKNOWN_CATEGORY"),
    ("add_expense", {"category": "food", "amount_inr": -1}, "AMOUNT_RANGE_0_TO_5000"),
    ("add_expense", {"category": "food", "amount_inr": True}, "FINITE_NUMBER_REQUIRED"),
    ("add_expense", {"category": "food", "amount_inr": "50"}, "FINITE_NUMBER_REQUIRED"),
    ("add_expense", {"category": "food", "amount_inr": float("nan")}, "FINITE_NUMBER_REQUIRED"),
    ("add_expense", {"category": "food", "amount_inr": 10.001}, "AT_MOST_TWO_DECIMAL_PLACES"),
    ("add_expense", {"category": "food", "amount_inr": 5001}, "AMOUNT_RANGE_0_TO_5000"),
    ("get_budget_summary", {"ignore_budget": True}, "ARGUMENT_CONTRACT"),
    ("pay_upi", {}, "UNKNOWN_TOOL"),
]
for name, arguments, expected_error in rejection_cases:
    before = copy.deepcopy(validation_state.expenses)
    result = validation_state.execute(name, arguments)
    assert result.get("error") == expected_error and validation_state.expenses == before
    print("Expected rejection:", expected_error, "— ledger unchanged")
# Exact decimal amounts remain valid and are stored as integer paise.
assert validation_state.execute("add_expense", {"category": "learning", "amount_inr": 10.25})["ok"]
assert validation_state.summary()["total_inr"] == 170.25
print("PASS: 9 rejection cases and one exact-paise addition.")


## 7. Optional live model experiment

Complete the baseline first. Add **your own** `GEMINI_API_KEY` in Colab Secrets,
enable notebook access, set `USE_LIVE_MODEL = True` above, and rerun the config
cell and this cell. The pinned SDK is `google-genai==2.12.0`; installation keeps
Colab's active `google-auth` constraints. Verify `gemini-3.5-flash-lite` availability
and account access before class. A physical request times out at 60 seconds and
has at most one automatic transient retry. 429/authentication errors do not retry.

If pending, inspect `live_agent.trace` and rerun this cell to resume the same turn.
It preserves counters and completed actions. To intentionally start afresh, call
`live_agent.reset()`; the request budget is still consumed. A completed live turn
is evidence only when the **stored summary and tool trace** match your request.


In [ ]:
# Live execution is opt-in; baseline fixture results never become live model answers.
LIVE_QUESTION = 'Read my budget, add INR 80 for food, then show the updated summary.'
live_reply = None
if not USE_LIVE_MODEL:
    print("Live model disabled. Baseline Run all requires no key, SDK install or network.")
else:
    # Rerunning this cell retains the existing simulation, history and request budget.
    existing = globals().get("live_agent")
    if existing is None:
        live_agent = ToolAgent(ExpenseState(), GeminiSource(), EXPENSE_SCHEMAS, EXPENSE_SYSTEM,
                               approve_write=None)
    else:
        live_agent = existing
        live_agent.__class__ = ToolAgent
        live_agent.state.__class__ = ExpenseState
        print("Existing live state, trace, pending turn and request counter retained.")
    if live_agent.pending is not None:
        if live_agent.pending["question"] == LIVE_QUESTION and live_agent.can_resume:
            live_reply = show_attempt("Resume live response", live_agent.resume)
        else:
            print("PENDING: inspect the earlier turn and limits before another question.")
    elif live_agent.last_result is None:
        live_reply = show_attempt("Live response", lambda: live_agent.run(LIVE_QUESTION))
    else:
        print("This live example already completed; inspect its trace before a new request.")
    live_agent.show_trace()
    print("Physical requests used:", live_agent.requests_used, "Pending:", live_agent.pending is not None)


## 8. Student changes — not implemented in the supplied baseline

Make all three changes in your own copy. Keep the baseline cells passing.

1. **Category rule:** add `printing` to both the category allowlist and the tool
   declaration. Record ₹12.50; assert its category total and exact paise value.
2. **Budget tool:** implement and declare `set_budget(budget_inr)` for positive
   finite amounts up to ₹10,000 and at most two decimals. Reject `True`, 0 and
   extra arguments; after setting ₹200 the baseline total ₹240 must show
   `over_budget=True` and `remaining_inr=-40`.
3. **Correction tool:** add `remove_expense(expense_id)` for an existing synthetic
   ID. Reject an unknown ID and a repeated removal without editing state. Use
   a new fixture decision sequence and prove only the intended row disappears.

**Checkpoints:** show an accepted tool trace, each new rejection, and before/after
ledger snapshots. Explain why a model's arithmetic or a final sentence is not
sufficient evidence that a write occurred. Optional live testing adds evidence;
an unavailable live provider does not erase a completed offline baseline.


### Your extension workspace
This intentionally unfinished cell changes nothing. Replace the comments with your implementation and add your own checks.


In [ ]:
# TODO 1: implement printing in the allowlist and declaration; check INR 12.50.
# TODO 2: implement set_budget with Python validation and an explicit tool schema.
# TODO 3: implement remove_expense and demonstrate missing/repeated-ID rejection.
print("Student extension workspace: 3 changes still to implement.")


## 9. Baseline receipt
This reports only supplied baseline behavior. Complete the three student changes before submitting your practice work.


In [ ]:
# This receipt describes the completed supplied baseline, not unfinished student work.
baseline_evidence = {"mode": expense_agent.source.mode, "synthetic": True,
    "baseline_complete": expense_agent.last_result is not None and expense_agent.pending is None,
    "tool_calls": len(expense_agent.trace), "decision_requests": expense_agent.requests_used,
    "state": expense_state.summary(), "student_extensions_complete": False}
print(json.dumps(baseline_evidence, indent=2))
print("Keep your own extension trace and before/after state as submission evidence.")


## Source and provenance

Independently written classroom adaptation of the tool declaration → model choice →
tool result pattern demonstrated in the official
[OpenAI Agents Python basic tools example](https://github.com/openai/openai-agents-python/blob/main/examples/basic/tools.py)
and [repository README](https://github.com/openai/openai-agents-python/blob/main/README.md).
Those sources were read on 4 November 2026; upstream code was not executed or copied.
This notebook uses manual dispatch and optional Google Gen AI, not the OpenAI SDK.
No IBM, client, real student, bank or college operational data is included.
